# Performance Prediction Agent — Model Training Notebook

This notebook does three things, in order:

1. **Generate** a synthetic dataset (since real run history in `manager_feedback.jsonl` is too small to train on yet)
2. **Verify** the dataset is logically correct (not just randomly generated) — prints correlation checks you can read yourself
3. **Train** two models on it: a duration regressor and an outcome classifier, then save them as `.pkl` files

**Run all cells top to bottom.** At the end you'll have:
- `synthetic_runs.csv` — the dataset
- `models/duration_regressor.pkl`
- `models/outcome_classifier.pkl`
- `models/feature_encoder.pkl`
- `models/metrics.json` — honest accuracy numbers, not cherry-picked

Copy the `models/` folder into your `performance_prediction_agent/` directory when done.

## 0. Setup

In [1]:
!pip install -q scikit-learn pandas numpy joblib --break-system-packages 2>/dev/null || pip install -q scikit-learn pandas numpy joblib

In [2]:
import os
import json
import numpy as np
import pandas as pd
import joblib

np.random.seed(42)
os.makedirs("models", exist_ok=True)
print("Setup complete.")

Setup complete.


## 1. Generate the Synthetic Dataset

We simulate realistic pipeline runs by encoding the same physics the formula-based
Performance Prediction Agent already uses (critical path duration, complexity,
contention, correction factors), then add controlled noise to represent real-world
variance.

**Why synthetic data at all?** Real run history (`manager_feedback.jsonl`) currently
has only a handful of rows — nowhere near enough to train a model. This dataset is
a stand-in until enough real runs accumulate.

**Key design choices (so you know what's encoded, not hidden):**
- `risk_score` is built from 5 weighted factors: file size, stage count, parallelism,
  Resource Agent correction-factor uncertainty, and a `network_quality` latent variable
  representing real-world unpredictability (Azure throttling, cold starts) that the
  plan itself can't predict.
- Outcome labels (`success`/`slowdown`/`failure`) are derived directly from `risk_score`
  thresholds, not from random noise — this is what makes the dataset *learnable*.
- `copy_correction`/`notebook_correction` deliberately show ~0 raw correlation with risk
  by design — what matters is their *deviation* from 1.0, not their raw value. We verify
  this distinction explicitly in step 2.

In [3]:
N_SAMPLES = 20000

def generate_dataset(n=N_SAMPLES) -> pd.DataFrame:
    rows = []

    for _ in range(n):
        # Stage composition
        stage_count = np.random.randint(1, 13)
        copy_stages = np.random.randint(0, stage_count + 1)
        notebook_stages = stage_count - copy_stages

        # File size and row count (log-normal — most files small, some huge)
        file_size_mb = np.clip(np.random.lognormal(mean=2.7, sigma=2.1), 0.1, 20000)
        row_count = int(file_size_mb * np.random.uniform(800, 1500))

        # Complexity bucket (matches Central Manager's own logic exactly)
        if file_size_mb > 100 or stage_count > 5:
            complexity = "high"
        elif file_size_mb > 10 or stage_count > 2:
            complexity = "medium"
        else:
            complexity = "low"

        # Parallelism: n_groups close to stage_count -> sequential (low parallel_ratio)
        #              n_groups close to 1            -> parallel   (high parallel_ratio)
        n_groups = np.random.randint(1, stage_count + 1)
        parallel_ratio = round(1 - (n_groups / stage_count), 3)

        # Resource Agent's own baseline estimate
        copy_time = copy_stages * np.random.uniform(40, 90)
        transform_count = np.random.randint(0, 6)
        agg_count = np.random.randint(0, 4)
        notebook_time = notebook_stages * (
            120 + transform_count * 3 + agg_count * 10 + (row_count / 50000)
        )
        baseline_s = copy_time + notebook_time
        baseline_s = baseline_s * (1 - parallel_ratio * 0.4)
        baseline_s = max(60, baseline_s)

        # Resource Agent correction factors (simulated self-correction drift)
        copy_correction = np.random.normal(1.0, 0.18)
        notebook_correction = np.random.normal(1.0, 0.22)
        copy_share = copy_stages / max(stage_count, 1)
        notebook_share = 1 - copy_share
        correction_blend = (copy_correction * copy_share) + (notebook_correction * notebook_share)

        resource_estimate_s = baseline_s * np.random.normal(1.0, 0.1)
        corrected_baseline_s = baseline_s * correction_blend

        # Latent external factor: real failures partly caused by things the
        # plan can't predict (network/cluster variance on the day)
        network_quality = np.random.beta(a=5, b=2)
        network_penalty = (1 - network_quality) * 0.25

        # Risk score from features that correlate with real-world causes
        correction_deviation = np.clip(abs(correction_blend - 1.0) / 0.4, 0, 1)
        risk_score = (
            0.28 * np.clip(file_size_mb / 2000, 0, 1)
            + 0.18 * np.clip(stage_count / 12, 0, 1)
            + 0.14 * (1 - parallel_ratio)
            + 0.18 * correction_deviation
            + 0.22 * network_penalty / 0.25
        )
        risk_score = np.clip(risk_score + np.random.normal(0, 0.06), 0, 1)

        noise_factor = np.random.lognormal(mean=0.0, sigma=0.15)
        chaos_multiplier = 1.0 + risk_score * np.random.uniform(2.0, 5.0)
        actual_duration_s = corrected_baseline_s * noise_factor * chaos_multiplier
        actual_duration_s = max(30, actual_duration_s)

        if risk_score >= 0.58:
            outcome = "failure"
        elif risk_score >= 0.38:
            outcome = "slowdown"
        else:
            outcome = "success"

        rows.append({
            "stage_count": stage_count,
            "copy_stages": copy_stages,
            "notebook_stages": notebook_stages,
            "file_size_mb": round(file_size_mb, 2),
            "row_count": row_count,
            "complexity": complexity,
            "n_execution_groups": n_groups,
            "parallel_ratio": parallel_ratio,
            "transform_count": transform_count,
            "agg_count": agg_count,
            "copy_correction": round(copy_correction, 3),
            "notebook_correction": round(notebook_correction, 3),
            "resource_estimate_s": round(resource_estimate_s, 1),
            "baseline_s": round(baseline_s, 1),
            "network_quality": round(network_quality, 3),
            "actual_duration_s": round(actual_duration_s, 1),
            "outcome": outcome,
        })

    return pd.DataFrame(rows)

df = generate_dataset()
df.to_csv("synthetic_runs.csv", index=False)
print(f"Generated {len(df)} rows -> synthetic_runs.csv")
print()
print("Outcome distribution:")
print(df["outcome"].value_counts())
print()
print("Complexity distribution:")
print(df["complexity"].value_counts())

Generated 20000 rows -> synthetic_runs.csv

Outcome distribution:
outcome
success     14363
slowdown     5351
failure       286
Name: count, dtype: int64

Complexity distribution:
complexity
high      13113
medium     5461
low        1426
Name: count, dtype: int64


## 2. Verify the Dataset Is Logically Correct

This is the step that answers "is the data logically correct or randomly generated?"
We check that every feature correlates with the outcome in the direction it *should*,
based on real-world reasoning — not just trust the generator blindly.

**What to look for below:**
- `file_size_mb`, `stage_count` should be **positive** (bigger/more complex = riskier)
- `parallel_ratio`, `network_quality` should be **negative** (more parallel / better network = safer)
- `correction_deviation` should be **positive** (uncertain Resource Agent estimates = riskier)
- Raw `copy_correction`/`notebook_correction` should be **near zero** — by design, only their
  *deviation* from 1.0 matters, not their raw sign
- `row_count` vs `file_size_mb` correlation should be **near 1.0** (near-linear, as it should be)
- Duplicate rows should be **0 or near 0**

If any of these come out backwards, the dataset has a real bug — don't proceed to training.

In [4]:
def verify_logical_correctness(df: pd.DataFrame):
    print("=" * 60)
    print("LOGICAL CORRECTNESS CHECKS")
    print("=" * 60)

    print(f"\nDuplicate rows: {df.duplicated().sum()} (should be 0 or near 0)")

    df["is_risky"] = (df["outcome"] != "success").astype(int)
    df["correction_deviation"] = (
        (df["copy_correction"] * df["copy_stages"] + df["notebook_correction"] * df["notebook_stages"])
        / df["stage_count"].clip(lower=1)
    ).sub(1.0).abs()

    print("\nCorrelation of each feature with risk (failure/slowdown):")
    print("  Expected: file_size_mb (+), stage_count (+), parallel_ratio (-),")
    print("  network_quality (-), correction_deviation (+), raw corrections (~0)")
    print()
    for col in [
        "stage_count", "file_size_mb", "parallel_ratio",
        "copy_correction", "notebook_correction",
        "correction_deviation", "network_quality", "n_execution_groups",
    ]:
        corr = df[col].corr(df["is_risky"])
        flag = ""
        print(f"    {col:22s}: {corr:+.3f}")

    print("\nRow count vs file size correlation (should be ~0.95+):")
    print(f"    {df[['file_size_mb', 'row_count']].corr().iloc[0, 1]:.3f}")

    print("\nFile size distribution (MB):")
    print(df["file_size_mb"].describe().to_string())

    df.drop(columns=["is_risky", "correction_deviation"], inplace=True)
    print("\n" + "=" * 60)

verify_logical_correctness(df)

LOGICAL CORRECTNESS CHECKS

Duplicate rows: 0 (should be 0 or near 0)

Correlation of each feature with risk (failure/slowdown):
  Expected: file_size_mb (+), stage_count (+), parallel_ratio (-),
  network_quality (-), correction_deviation (+), raw corrections (~0)

    stage_count           : +0.270
    file_size_mb          : +0.189
    parallel_ratio        : -0.179
    copy_correction       : -0.001
    notebook_correction   : -0.013
    correction_deviation  : +0.337
    network_quality       : -0.266
    n_execution_groups    : +0.385

Row count vs file size correlation (should be ~0.95+):
    0.988

File size distribution (MB):
count    20000.000000
mean       129.485189
std        642.211516
min          0.100000
25%          3.570000
50%         14.780000
75%         60.090000
max      20000.000000



**Read the output above before continuing.** If the signs don't match the expected
directions listed, stop here and fix the generator in the cell above — don't train
a model on data you haven't verified.

## 3. Train the Models

Two models, trained independently:

1. **Duration regressor** (GradientBoostingRegressor) — predicts `actual_duration_s`.
   Trained on `log1p(duration)` since durations are right-skewed (most runs are short,
   a few are very long) — this is standard practice for skewed targets.
2. **Outcome classifier** (RandomForestClassifier) — predicts `success`/`slowdown`/`failure`.
   Uses `class_weight="balanced"` because failure is intentionally rare (~1-2% of data,
   matching real-world rarity) — without this the model would just always predict
   "success" and look falsely accurate.

In [5]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, mean_absolute_error, r2_score,
    balanced_accuracy_score, f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

FEATURE_COLS = [
    "stage_count", "copy_stages", "notebook_stages",
    "file_size_mb", "row_count",
    "n_execution_groups", "parallel_ratio",
    "transform_count", "agg_count",
    "copy_correction", "notebook_correction",
    "resource_estimate_s", "baseline_s",
    "network_quality",
    "complexity_encoded",
]

encoder = LabelEncoder()
df["complexity_encoded"] = encoder.fit_transform(df["complexity"])

X = df[FEATURE_COLS]
y_duration = df["actual_duration_s"]
y_duration_log = np.log1p(y_duration)
y_outcome = df["outcome"]

X_train, X_test, yd_train, yd_test, ydlog_train, ydlog_test, yo_train, yo_test = train_test_split(
    X, y_duration, y_duration_log, y_outcome,
    test_size=0.2, random_state=42, stratify=y_outcome
)

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
print(f"Train outcome distribution: {yo_train.value_counts().to_dict()}")

Train: 16000 rows | Test: 4000 rows
Train outcome distribution: {'success': 11490, 'slowdown': 4281, 'failure': 229}


In [6]:
# Duration regressor
print("Training duration regressor...")
reg = GradientBoostingRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42,
)
reg.fit(X_train, ydlog_train)
ydlog_pred = reg.predict(X_test)
yd_pred = np.expm1(ydlog_pred)

mae = mean_absolute_error(yd_test, yd_pred)
r2 = r2_score(yd_test, yd_pred)
print(f"  MAE: {mae:.1f}s")
print(f"  R^2: {r2:.3f}")

Training duration regressor...
  MAE: 232.9s
  R^2: 0.867


In [7]:
# Outcome classifier
print("Training outcome classifier...")
clf = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=3,
    class_weight="balanced",
    random_state=42, n_jobs=-1,
)
clf.fit(X_train, yo_train)
yo_pred = clf.predict(X_test)

acc = accuracy_score(yo_test, yo_pred)
bal_acc = balanced_accuracy_score(yo_test, yo_pred)
macro_f1 = f1_score(yo_test, yo_pred, average="macro")
report = classification_report(yo_test, yo_pred, output_dict=True)

print(f"  Accuracy:          {acc:.3f}")
print(f"  Balanced accuracy:  {bal_acc:.3f}  <- this matters more than raw accuracy")
print(f"  Macro F1:           {macro_f1:.3f}")
print()
print(classification_report(yo_test, yo_pred))

Training outcome classifier...
  Accuracy:          0.774
  Balanced accuracy:  0.636  <- this matters more than raw accuracy
  Macro F1:           0.633

              precision    recall  f1-score   support

     failure       0.46      0.40      0.43        57
    slowdown       0.56      0.69      0.62      1070
     success       0.88      0.81      0.85      2873

    accuracy                           0.77      4000
   macro avg       0.64      0.64      0.63      4000
weighted avg       0.79      0.77      0.78      4000



**Read these numbers honestly before moving on.** `failure` is the rare class —
expect lower precision/recall on it than `success`. If `balanced_accuracy` is close
to the overall `accuracy`, the model is actually distinguishing classes, not just
defaulting to the majority class. If `failure` recall is 0.00, something is broken —
check the class distribution in the previous cell.

In [8]:
# Feature importance — sanity check on what's actually driving predictions
print("Top features driving duration predictions:")
importances = sorted(zip(FEATURE_COLS, reg.feature_importances_), key=lambda x: -x[1])
for name, imp in importances[:7]:
    print(f"  {name:22s}: {imp:.3f}")

Top features driving duration predictions:
  baseline_s            : 0.908
  notebook_correction   : 0.026
  copy_correction       : 0.018
  resource_estimate_s   : 0.013
  n_execution_groups    : 0.010
  copy_stages           : 0.008
  network_quality       : 0.006


## 4. Save the Models

These are saved locally as `.pkl` files. Copy the `models/` folder into your
`performance_prediction_agent/` directory in your codebase when this notebook finishes.

In [9]:
joblib.dump(reg, "models/duration_regressor.pkl")
joblib.dump(clf, "models/outcome_classifier.pkl")
joblib.dump(encoder, "models/feature_encoder.pkl")

metrics = {
    "duration_regressor": {
        "mae_seconds": round(mae, 2),
        "r2_score": round(r2, 4),
    },
    "outcome_classifier": {
        "accuracy": round(acc, 4),
        "balanced_accuracy": round(bal_acc, 4),
        "macro_f1": round(macro_f1, 4),
        "classification_report": report,
    },
    "feature_columns": FEATURE_COLS,
    "n_training_samples": len(X_train),
    "n_test_samples": len(X_test),
}

with open("models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:")
print("  models/duration_regressor.pkl")
print("  models/outcome_classifier.pkl")
print("  models/feature_encoder.pkl")
print("  models/metrics.json")

Saved:
  models/duration_regressor.pkl
  models/outcome_classifier.pkl
  models/feature_encoder.pkl
  models/metrics.json


## 5. Next Steps

1. Download the `models/` folder from this notebook's working directory
2. Copy it into `performance_prediction_agent/models/` in your codebase
3. Place `ml_predictor.py` (provided separately) into `performance_prediction_agent/`
4. Apply the patch to `performance_agent.py` so it tries the ML model first,
   falling back to the existing formula if model files are missing or inference fails
5. Restart your backend and test via `/api/performance-prediction/predict`

**Honest reminder for your teammate:** this model is trained on synthetic data, not
real pipeline behavior. It encodes reasonable assumptions about what makes a pipeline
risky, but it has never seen a real Azure/Databricks run. Retrain it on real
`manager_feedback.jsonl` data once you have 50-100+ real runs — that's when predictions
will actually reflect your system's real-world behavior instead of a simulation.